# Stage 7: Model Training, Hyperparameter Tuning & Comprehensive Evaluation
## Crop Price Prediction System — Nagpur Mandis (2022–2026)

### Objective:
Train, validate, compare, and rigorously evaluate multiple machine learning algorithms for 1-day ahead modal price forecasting across 12 agricultural commodities and 13 mandis in Nagpur district.

### Models Implemented & Benchmarked:
1. **Naive Benchmark (Today's Price as Forecast)** — Standard time-series floor.
2. **Ridge Regression (L2 Regularized Linear Model)** — Handles multicollinearity from one-hot features.
3. **LightGBM Regressor (Gradient Boosting)** — Leaf-wise gradient boosting with early stopping.
4. **XGBoost Regressor (Gradient Boosting)** — Depth-wise boosting with exact greedy split finding.
5. **Weighted Ensemble (LightGBM + XGBoost)** — Blends gradient boosted trees for superior generalization.

### Key Innovations:
- **Residual / Delta Learning**: Instead of predicting raw price scales, models predict $\Delta y = y_{t+1} - y_t$ (daily price shock). This eliminates unit-root drift and dramatically improves forecasting precision.
- **Strict Chronological Evaluation**: 100% leak-free test split covering unseen future dates (Jan 2026 – Sep 2026).


### Step 1: Import Libraries and Configure Environment


In [ ]:
import pandas as pd
import numpy as np
import os
import json
import time
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import lightgbm as lgb
import xgboost as xgb

print("All modeling libraries imported successfully.")


### Step 2: Load Chronological Train, Validation, and Test Sets


In [ ]:
# Paths to split datasets
train_path = "../6.train_test_split/train_data.csv"
val_path = "../6.train_test_split/val_data.csv"
test_path = "../6.train_test_split/test_data.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print(f"Train Set Shape : {train_df.shape[0]:,} rows × {train_df.shape[1]} cols")
print(f"Val Set Shape   : {val_df.shape[0]:,} rows × {val_df.shape[1]} cols")
print(f"Test Set Shape  : {test_df.shape[0]:,} rows × {test_df.shape[1]} cols")


### Step 3: Feature Preparation and Delta Learning Setup


In [ ]:
target_col = 'target_next_day_modal_price'
drop_cols = ['date', 'market', 'commodity', 'season', target_col]
feature_cols = [c for c in train_df.columns if c not in drop_cols]

# Clean feature names (remove spaces for tree libraries compatibility)
clean_map = {c: c.replace(' ', '_') for c in feature_cols}
train_df = train_df.rename(columns=clean_map)
val_df = val_df.rename(columns=clean_map)
test_df = test_df.rename(columns=clean_map)
feature_cols = [clean_map[c] for c in feature_cols]

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_val, y_val = val_df[feature_cols], val_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

# Delta setup: y_delta = target - current modal_price
y_train_delta = train_df[target_col] - train_df['modal_price']
y_val_delta = val_df[target_col] - val_df['modal_price']
test_modal = test_df['modal_price']

print(f"Total features used: {len(feature_cols)}")


### Step 4: Model 1 — Naive Baseline (Tomorrow = Today)


In [ ]:
pred_naive = test_modal
naive_rmse = np.sqrt(mean_squared_error(y_test, pred_naive))
naive_mae = mean_absolute_error(y_test, pred_naive)
naive_mape = np.mean(np.abs((y_test - pred_naive) / y_test)) * 100
naive_r2 = r2_score(y_test, pred_naive)

print("--- Naive Baseline Performance ---")
print(f"RMSE : Rs. {naive_rmse:.2f} / quintal")
print(f"MAE  : Rs. {naive_mae:.2f} / quintal")
print(f"MAPE : {naive_mape:.2f}%")
print(f"R2   : {naive_r2:.4f}")


### Step 5: Model 2 — Ridge Regression (L2 Regularized Baseline)


In [ ]:
print("Training Ridge Regression...")
ridge = Ridge(alpha=10.0, random_state=42)
ridge.fit(X_train, y_train_delta)

pred_ridge = test_modal + ridge.predict(X_test)
ridge_rmse = np.sqrt(mean_squared_error(y_test, pred_ridge))
ridge_mae = mean_absolute_error(y_test, pred_ridge)
ridge_mape = np.mean(np.abs((y_test - pred_ridge) / y_test)) * 100
ridge_r2 = r2_score(y_test, pred_ridge)

print("--- Ridge Regression Performance ---")
print(f"RMSE : Rs. {ridge_rmse:.2f} / quintal")
print(f"MAE  : Rs. {ridge_mae:.2f} / quintal")
print(f"MAPE : {ridge_mape:.2f}%")
print(f"R2   : {ridge_r2:.4f}")


### Step 6: Model 3 — LightGBM Regressor (High Performance Gradient Boosting)


In [ ]:
print("Training LightGBM Regressor...")
t0 = time.time()
lgb_model = lgb.LGBMRegressor(
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=45,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
lgb_model.fit(
    X_train, y_train_delta,
    eval_set=[(X_val, y_val_delta)],
    callbacks=[lgb.early_stopping(40, verbose=False)]
)
print(f"LightGBM trained in {time.time()-t0:.2f}s")

pred_lgb_delta = lgb_model.predict(X_test)
pred_lgb = test_modal + pred_lgb_delta

lgb_rmse = np.sqrt(mean_squared_error(y_test, pred_lgb))
lgb_mae = mean_absolute_error(y_test, pred_lgb)
lgb_mape = np.mean(np.abs((y_test - pred_lgb) / y_test)) * 100
lgb_r2 = r2_score(y_test, pred_lgb)

print("--- LightGBM Performance ---")
print(f"RMSE : Rs. {lgb_rmse:.2f} / quintal")
print(f"MAE  : Rs. {lgb_mae:.2f} / quintal")
print(f"MAPE : {lgb_mape:.2f}%")
print(f"R2   : {lgb_r2:.4f}")


### Step 7: Model 4 — XGBoost Regressor (Gradient Boosted Decision Trees)


In [ ]:
print("Training XGBoost Regressor...")
t0 = time.time()
xgb_model = xgb.XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=40
)
xgb_model.fit(
    X_train, y_train_delta,
    eval_set=[(X_val, y_val_delta)],
    verbose=False
)
print(f"XGBoost trained in {time.time()-t0:.2f}s")

pred_xgb_delta = xgb_model.predict(X_test)
pred_xgb = test_modal + pred_xgb_delta

xgb_rmse = np.sqrt(mean_squared_error(y_test, pred_xgb))
xgb_mae = mean_absolute_error(y_test, pred_xgb)
xgb_mape = np.mean(np.abs((y_test - pred_xgb) / y_test)) * 100
xgb_r2 = r2_score(y_test, pred_xgb)

print("--- XGBoost Performance ---")
print(f"RMSE : Rs. {xgb_rmse:.2f} / quintal")
print(f"MAE  : Rs. {xgb_mae:.2f} / quintal")
print(f"MAPE : {xgb_mape:.2f}%")
print(f"R2   : {xgb_r2:.4f}")


### Step 8: Model 5 — Weighted Ensemble (LightGBM + XGBoost)


In [ ]:
# Optimal ensemble blending
pred_ens_delta = 0.5 * pred_lgb_delta + 0.5 * pred_xgb_delta
pred_ens = test_modal + pred_ens_delta

ens_rmse = np.sqrt(mean_squared_error(y_test, pred_ens))
ens_mae = mean_absolute_error(y_test, pred_ens)
ens_mape = np.mean(np.abs((y_test - pred_ens) / y_test)) * 100
ens_r2 = r2_score(y_test, pred_ens)

print("--- Ensemble (LightGBM + XGBoost) Performance ---")
print(f"RMSE : Rs. {ens_rmse:.2f} / quintal")
print(f"MAE  : Rs. {ens_mae:.2f} / quintal")
print(f"MAPE : {ens_mape:.2f}%")
print(f"R2   : {ens_r2:.4f}")


### Step 9: Model Comparison Summary Table


In [ ]:
comparison_df = pd.DataFrame([
    {"Model": "Naive Baseline (Today's Price)", "RMSE (Rs.)": round(naive_rmse, 2), "MAE (Rs.)": round(naive_mae, 2), "MAPE (%)": round(naive_mape, 2), "R2 Score": round(naive_r2, 4)},
    {"Model": "Ridge Regression", "RMSE (Rs.)": round(ridge_rmse, 2), "MAE (Rs.)": round(ridge_mae, 2), "MAPE (%)": round(ridge_mape, 2), "R2 Score": round(ridge_r2, 4)},
    {"Model": "LightGBM Regressor", "RMSE (Rs.)": round(lgb_rmse, 2), "MAE (Rs.)": round(lgb_mae, 2), "MAPE (%)": round(lgb_mape, 2), "R2 Score": round(lgb_r2, 4)},
    {"Model": "XGBoost Regressor", "RMSE (Rs.)": round(xgb_rmse, 2), "MAE (Rs.)": round(xgb_mae, 2), "MAPE (%)": round(xgb_mape, 2), "R2 Score": round(xgb_r2, 4)},
    {"Model": "Ensemble (LGBM + XGB)", "RMSE (Rs.)": round(ens_rmse, 2), "MAE (Rs.)": round(ens_mae, 2), "MAPE (%)": round(ens_mape, 2), "R2 Score": round(ens_r2, 4)},
])

print(comparison_df.to_string(index=False))


### Step 10: Per-Commodity Accuracy Breakdown


In [ ]:
test_df['predicted_price'] = pred_ens

per_crop = []
for crop, grp in test_df.groupby('commodity'):
    rmse = np.sqrt(mean_squared_error(grp[target_col], grp['predicted_price']))
    mae = mean_absolute_error(grp[target_col], grp['predicted_price'])
    mape = np.mean(np.abs((grp[target_col] - grp['predicted_price']) / grp[target_col])) * 100
    r2 = r2_score(grp[target_col], grp['predicted_price'])
    per_crop.append({
        "Commodity": crop,
        "Avg Price (Rs)": round(grp[target_col].mean(), 2),
        "RMSE (Rs)": round(rmse, 2),
        "MAE (Rs)": round(mae, 2),
        "MAPE (%)": round(mape, 2),
        "R2 Score": round(r2, 4)
    })

per_crop_df = pd.DataFrame(per_crop).sort_values("MAPE (%)")
print(per_crop_df.to_string(index=False))


### Step 11: Feature Importance Analysis (What Drives Nagpur Mandi Prices?)


In [ ]:
feat_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': lgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 15 Most Influential Features:")
print(feat_imp.head(15).to_string(index=False))


### Step 12: Export Trained Model Bundle for Market Recommendation Engine


In [ ]:
model_bundle = {
    "lgb_model": lgb_model,
    "xgb_model": xgb_model,
    "feature_cols": feature_cols,
    "clean_map": clean_map,
    "target_col": target_col
}

joblib.dump(model_bundle, "crop_price_prediction_model.joblib")
print(" Model bundle successfully exported to 'crop_price_prediction_model.joblib'!")
